# More accurate MiniRocket, with faster training and prediction

CinCECGTorso is a compact UCR training set where temporal organization matters: 40 training series, 1,380 test series and 1,639 time steps. It is one of the clearest ROMAN gains in the paper.

This example compares MiniRocket on the original input with the same classifier preceded by ROMAN at S=4. Both use 10,000 kernels and the same ridge regularization grid. The raw baseline and training-fitted channel normalization in ROMAN follow the paper's recorded MiniRocket protocol.

Install the examples dependencies with `pip install -e ".[examples]"`. Dataset loading may download the public archive on the first run. S=4 is fixed here to illustrate this favorable dataset; choose S using validation data on a new task.

In [1]:
from pathlib import Path
import sys

# Support running the notebook from the repository root or notebooks/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src" / "roman" / "__init__.py").exists())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "notebooks"))

from IPython.display import display
from example_utils import (load_dataset, make_model, benchmark_single,
                           benchmark_ensembles, display_single, environment)


In [2]:
DATASET = "CinCECGTorso"
DATA_ROOT = None  # Optional local sktime .ts cache; ROMAN_DATA_ROOT is also supported.
X_train, y_train, X_test, y_test = load_dataset(DATASET, DATA_ROOT)
print("Training:", X_train.shape, "  Test:", X_test.shape)

model = make_model(scale=4, seed=0)
model

Training: (40, 1, 1639)   Test: (1380, 1, 1639)


,steps,"[('romanoperator', ...), ('minirocketmultivariate', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,alpha,0.5
,min_timesteps_per_channel,32
,normalization,True
,S_max,None
,S,4
,max_pseudochannels,None
,N,None


## Measure the complete pipelines

Training includes normalization and routing, training-feature extraction and ridge fitting. Inference means predicting the entire test set from its original series, including ROMAN and MiniRocket feature extraction. Each prediction recomputes those features.

Both paths are warmed before measurement and run on one compute thread. Compilation and data loading are excluded. We average accuracy over seeds 0–2, report the median training time across those fits, and take five warm inference measurements per fitted model.

In [3]:
details, summary = benchmark_single(
    X_train, y_train, X_test, y_test,
    seeds=(0, 1, 2), scales=(1, 4), repeats=5,
)
display(display_single(summary))

S=1, seed=0: accuracy=79.78%


S=4, seed=0: accuracy=96.81%


S=4, seed=1: accuracy=95.43%


S=1, seed=1: accuracy=81.81%


S=1, seed=2: accuracy=80.07%


S=4, seed=2: accuracy=96.74%


,Model,Test accuracy,Training time,Inference time
0,MiniRocket (S=1),80.56%,0.317 s,6.625 s
1,ROMAN S=4 + MiniRocket,96.33%,0.149 s,3.169 s


In [4]:
baseline = summary.set_index("scale").loc[1]
routed = summary.set_index("scale").loc[4]
print(f"Accuracy gain: {100 * (routed.test_accuracy - baseline.test_accuracy):.2f} percentage points")
print(f"Training speedup: {baseline.training_s / routed.training_s:.2f}x")
print(f"Inference speedup: {baseline.inference_s / routed.inference_s:.2f}x")

display(details.drop(columns="inference_repetitions_s"))

Accuracy gain: 15.77 percentage points
Training speedup: 2.13x
Inference speedup: 2.09x


,scale,seed,test_accuracy,training_s,inference_s
0,1,0,0.797826,0.314940,6.608487
1,1,1,0.818116,0.317100,6.628553
2,1,2,0.800725,0.321047,6.624848
3,4,0,0.968116,0.153574,3.178710
4,4,1,0.954348,0.148130,3.168570
5,4,2,0.967391,0.148574,3.146797


## Reuse ROMAN with your own classifier

`make_model` uses the standard scikit-learn pipeline below. Replace the downstream classifier or choose a different S while keeping the same train/test workflow.

In [5]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import RidgeClassifierCV
from sktime.transformations.panel.rocket import MiniRocketMultivariate
from roman import RomanOperator

roman_model = make_pipeline(
    RomanOperator(S=4, alpha=0.5),
    MiniRocketMultivariate(num_kernels=10_000, random_state=0, n_jobs=1),
    RidgeClassifierCV(alphas=np.logspace(-3, 3, 10)),
)
# roman_model.fit(X_train, y_train)
# predictions = roman_model.predict(X_test)
roman_model

,steps,"[('romanoperator', ...), ('minirocketmultivariate', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,alpha,0.5
,min_timesteps_per_channel,32
,normalization,True
,S_max,None
,S,4
,max_pseudochannels,None
,N,None


In [6]:
environment()

{'python': '3.12.14',
 'platform': 'macOS-27.2-arm64-arm-64bit',
 'machine': 'arm64',
 'threads': 1,
 'packages': {'numpy': '2.3.5',
  'pandas': '2.3.3',
  'scipy': '1.17.1',
  'scikit-learn': '1.7.2',
  'sktime': '1.1.0',
  'numba': '0.63.1',
  'threadpoolctl': '3.6.0'}}